# E-Commerce FAQ Bot
**Domain:** Online retail customer support  
**User:** Online shoppers asking about products, shipping, returns, and policies  
**Success criteria:** Correctly routes and answers 80%+ of common support queries using only the knowledge base  
**Tool:** datetime (to answer "what date is today" queries)

---
**Setup required (Colab Secrets):**
- `GROQ_API_KEY` — required for the LLM
- `NGROK_AUTH_TOKEN` — required to expose the Streamlit UI
Run all cells top-to-bottom. The last cell prints your public URL.

## Part 1 — Install & Imports

In [1]:
# Uninstall conflicting packages first
!pip uninstall -y google-adk opentelemetry-api opentelemetry-sdk \
    opentelemetry-exporter-otlp-proto-http -q

!pip install -q \
    chromadb \
    sentence-transformers \
    langchain-community \
    langchain-groq \
    langgraph \
    streamlit \
    pyngrok \
    langchain-huggingface \
    ragas

print('All packages installed.')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.41.0 which is incompatible.
All packages installed.


In [26]:
import os, re, uuid, datetime, subprocess, time
from typing import TypedDict, List, Optional
from google.colab import userdata
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

def _secret(key):
    try:
        return userdata.get(key)
    except Exception:
        return None

GROQ_API_KEY  = _secret('GROQ_API_KEY')
NGROK_TOKEN   = _secret('NGROK_AUTH')

if GROQ_API_KEY:
    os.environ['GROQ_API_KEY'] = GROQ_API_KEY
    print('GROQ_API_KEY loaded.')
else:
    print('WARNING: GROQ_API_KEY not found. Add it to Colab Secrets.')

if NGROK_TOKEN:
    print('NGROK_AUTH loaded.')
else:
    print('WARNING: NGROK_AUTH_TOKEN not found. Add it to Colab Secrets.')

GROQ_API_KEY loaded.
NGROK_AUTH loaded.


## Part 1 — Knowledge Base (12 Documents)

In [29]:
DOCUMENTS = [
    # PRODUCTS
    {
        'id': 'doc_001',
        'topic': 'Wireless Headphones',
        'text': (
            'The ShopMax Wireless Headphones (Model WH-300) use Bluetooth 5.3 for a stable, '
            'low-latency connection up to 10 metres. Battery life is 20 hours on a single charge, '
            'with a 2-hour fast-charge via USB-C. The headphones include active noise cancellation (ANC), '
            'a built-in microphone for calls, and 40mm drivers for rich bass. Weight is 250g. '
            'They are foldable and come with a hard carrying case. Available colours: Black, White, Navy Blue. '
            'Price: Rs 1,999. Covered by a 1-year manufacturer warranty against hardware defects. '
            'Compatible with iOS, Android, Windows, and macOS. Not waterproof.'
        )
    },
    {
        'id': 'doc_002',
        'topic': 'Smart Watch',
        'text': (
            'The ShopMax Smart Watch (Model SW-200) tracks steps, calories, sleep, heart rate, '
            'and blood oxygen (SpO2). It has a 1.4-inch AMOLED display with always-on mode. '
            'Battery life is 7 days in normal mode, 3 days with always-on display enabled. '
            'It is IP68 waterproof — safe for swimming up to 50 metres. Compatible with iOS 12+ and Android 8+. '
            'Includes 20+ workout modes (running, cycling, yoga, swimming). '
            'Built-in GPS for outdoor runs. Supports notifications for calls, SMS, and apps. '
            'Strap is interchangeable (22mm standard). Price: Rs 2,999. 1-year warranty.'
        )
    },
    {
        'id': 'doc_003',
        'topic': 'Mechanical Keyboard',
        'text': (
            'The ShopMax Mechanical Keyboard (Model KB-75) is a tenkeyless (TKL) wired keyboard with '
            'Cherry MX Red switches — linear, quiet, and ideal for both typing and gaming. '
            'It has per-key RGB backlighting with 15 preset lighting modes. '
            'The aluminium top plate and PBT double-shot keycaps make it durable and resistant to shine. '
            'N-key rollover (NKRO) ensures every keypress is registered simultaneously. '
            'Connection: USB-A braided cable (1.8m). Layout: US ANSI. '
            'Compatible with Windows 10/11 and macOS. No driver software required. '
            'Price: Rs 3,499. 2-year warranty. Available in Space Grey only.'
        )
    },
    {
        'id': 'doc_004',
        'topic': 'Portable Bluetooth Speaker',
        'text': (
            'The ShopMax Blast Speaker (Model BS-50) delivers 20W stereo sound with dual 10W drivers '
            'and a passive radiator for deep bass. Battery lasts 12 hours at 70% volume. '
            'It is IPX7 waterproof — fully submersible up to 1 metre for 30 minutes. '
            'Bluetooth 5.0 range is up to 15 metres. Also supports wired AUX input (3.5mm). '
            'Has a built-in microphone for speakerphone calls. Charges in 3 hours via USB-C. '
            'Dimensions: 18cm x 7cm. Weight: 600g. Colours: Red, Black, Forest Green. '
            'Price: Rs 2,499. 1-year warranty. Not for underwater use beyond 1 metre.'
        )
    },
    {
        'id': 'doc_005',
        'topic': 'USB-C Fast Charger',
        'text': (
            'The ShopMax 65W USB-C GaN Charger (Model FC-65) supports USB Power Delivery 3.0 '
            'and charges laptops, tablets, and phones. It has two ports: one USB-C (65W max) and '
            'one USB-A (18W max, Quick Charge 3.0). The GaN (Gallium Nitride) technology makes it '
            '40% smaller than traditional chargers of the same wattage. '
            'Charges an iPhone 15 to 50% in 30 minutes. Charges a MacBook Air in under 2 hours. '
            'Universal voltage (100-240V) — suitable for international travel. '
            'Cable not included. Price: Rs 1,299. 1-year warranty. Colour: White.'
        )
    },
    {
        'id': 'doc_006',
        'topic': 'Laptop Stand',
        'text': (
            'The ShopMax ErgoRise Laptop Stand (Model LS-20) is made from anodised aluminium and '
            'supports laptops from 10 to 16 inches weighing up to 10kg. '
            'Height is adjustable across 6 levels (10cm to 20cm) and the angle can be set from 15 to 45 degrees. '
            'Folds flat in seconds for portability — packed size is 28cm x 22cm x 1.5cm, weight 500g. '
            'Silicone pads grip the desk and protect the laptop from scratches. '
            'Does not include a USB hub. Compatible with all brands (Apple, Dell, HP, Lenovo, Asus, etc.). '
            'Price: Rs 899. 1-year warranty. Available in Silver and Space Grey.'
        )
    },

    #POLICIES
    {
        'id': 'doc_007',
        'topic': 'Return Policy',
        'text': (
            'Customers may return any product within 7 days of delivery for a full refund, '
            'provided the item is unused, in its original packaging, and all accessories and manuals are included. '
            'To initiate a return, log in to your ShopMax account, go to Orders, select the item, and click Return. '
            'A pickup will be arranged within 1-2 business days at no cost to the customer. '
            'Items that are damaged due to customer misuse, have missing parts, or are returned after 7 days '
            'are not eligible for a refund. '
            'Opened software, downloaded digital products, and personalised/custom items cannot be returned. '
            'Refunds are processed within 5-7 business days of the return being received and verified.'
        )
    },
    {
        'id': 'doc_008',
        'topic': 'Shipping Policy',
        'text': (
            'ShopMax ships to all major cities and towns across India. '
            'Standard delivery takes 3-5 business days and is free for orders above Rs 499. '
            'For orders below Rs 499, a flat shipping fee of Rs 49 applies. '
            'Express delivery (1-2 business days) costs Rs 99 regardless of order value. '
            'Same-day delivery is available in Bangalore, Mumbai, Delhi, Hyderabad, and Chennai for orders placed before 11am. '
            'Same-day delivery costs Rs 149. '
            'Orders are dispatched Monday to Saturday (excluding public holidays). '
            'A tracking link is sent via SMS and email once the order ships. '
            'International shipping is not currently available.'
        )
    },
    {
        'id': 'doc_009',
        'topic': 'Refund Policy',
        'text': (
            'Once a returned item is received and verified by our warehouse team, the refund is processed '
            'within 5-7 business days. '
            'For orders paid by credit or debit card, the refund appears on the original card. '
            'For UPI and net banking payments, the refund is credited to the original payment account. '
            'Cash on Delivery orders are refunded via bank transfer — customers must provide their bank account number and IFSC code. '
            'ShopMax Wallet refunds (store credit) are processed within 24 hours. '
            'Customers receive an email confirmation when the refund is initiated. '
            'If the refund is not received within 7 business days, contact support with your order ID.'
        )
    },
    {
        'id': 'doc_010',
        'topic': 'Warranty Policy',
        'text': (
            'All ShopMax products include a manufacturer warranty covering hardware defects. '
            'Most products carry a 1-year warranty; the Mechanical Keyboard (Model KB-75) carries a 2-year warranty. '
            'Warranty claims must be submitted within the warranty period by contacting support at support@shopmax.in or calling 1800-123-4567 (toll-free). '
            'The warranty covers: dead-on-arrival (DOA) units, manufacturing defects, and component failure under normal use. '
            'The warranty does NOT cover: physical damage, water damage (unless the product is rated waterproof), '
            'damage from incorrect voltage, and damage from unauthorised repair. '
            'Proof of purchase (order ID or invoice) is required for all warranty claims. '
            'Approved warranty claims result in a free repair or replacement at ShopMax discretion.'
        )
    },
    {
        'id': 'doc_011',
        'topic': 'Payment Methods',
        'text': (
            'ShopMax accepts the following payment methods: '
            'Credit cards (Visa, Mastercard, Amex, RuPay), debit cards (all major banks), '
            'UPI (Google Pay, PhonePe, Paytm, BHIM, and any UPI app), '
            'net banking (50+ banks supported), Cash on Delivery (COD) for orders up to Rs 10,000, '
            'and ShopMax Wallet (store credit). '
            'EMI is available on orders above Rs 3,000 via eligible credit cards (3, 6, 9, and 12-month tenures). '
            'Buy Now Pay Later (BNPL) is available through ZestMoney and LazyPay for eligible customers. '
            'All transactions are secured with 256-bit SSL encryption. '
            'Prices shown are inclusive of GST (18% for electronics).'
        )
    },
    {
        'id': 'doc_012',
        'topic': 'Order Tracking and Cancellation',
        'text': (
            'Once an order is placed, a confirmation email and SMS are sent immediately. '
            'After dispatch, a tracking link from our courier partner (Delhivery, Ekart, or BlueDart) is shared via SMS and email. '
            'To track your order, visit shopmax.in/track or use the ShopMax app and enter your order ID. '
            'Orders can be cancelled for free within 12 hours of placement if not yet dispatched. '
            'To cancel, go to My Orders in your account and click Cancel Order. '
            'Orders that are already dispatched cannot be cancelled — you must wait for delivery and then raise a return request. '
            'Refunds for cancelled pre-dispatch orders are processed within 24 hours. '
            'For any issues with tracking or cancellation, contact support@shopmax.in or call 1800-123-4567.'
        )
    },
]

print(f'{len(DOCUMENTS)} documents loaded into knowledge base.')
for d in DOCUMENTS:
    print(f"  [{d['id']}] {d['topic']} — {len(d['text'])} chars")

12 documents loaded into knowledge base.
  [doc_001] Wireless Headphones — 586 chars
  [doc_002] Smart Watch — 556 chars
  [doc_003] Mechanical Keyboard — 586 chars
  [doc_004] Portable Bluetooth Speaker — 540 chars
  [doc_005] USB-C Fast Charger — 525 chars
  [doc_006] Laptop Stand — 560 chars
  [doc_007] Return Policy — 674 chars
  [doc_008] Shipping Policy — 606 chars
  [doc_009] Refund Policy — 640 chars
  [doc_010] Warranty Policy — 772 chars
  [doc_011] Payment Methods — 609 chars
  [doc_012] Order Tracking and Cancellation — 723 chars


## Part 1 — Build VectorDB & Verify Retrieval

In [30]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

def load_vectordb():
    embedding = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
    texts  = [d['text']  for d in DOCUMENTS]
    metas  = [{'topic': d['topic'], 'id': d['id']} for d in DOCUMENTS]
    db = Chroma.from_texts(texts=texts, embedding=embedding, metadatas=metas)
    return db

vectordb = load_vectordb()
print('VectorDB built successfully.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorDB built successfully.


In [32]:
test_queries = [
    'What is the return policy?',
    'How long does shipping take?',
    'Tell me about the wireless headphones',
    'How do I cancel my order?',
    'What payment methods are accepted?',
]

print('Retrieval Verification \n')
for q in test_queries:
    docs = vectordb.similarity_search(q, k=2)
    topics = [d.metadata['topic'] for d in docs]
    print(f'Q: {q}')
    print(f'   Retrieved: {topics}\n')

print('Retrieval verified. Proceeding to graph construction.')

Retrieval Verification 

Q: What is the return policy?
   Retrieved: ['Return Policy', 'Return Policy']

Q: How long does shipping take?
   Retrieved: ['Return Policy', 'Return Policy']

Q: Tell me about the wireless headphones
   Retrieved: ['Wireless Headphones', 'Wireless Headphones']

Q: How do I cancel my order?
   Retrieved: ['Order Tracking and Cancellation', 'Order Tracking and Cancellation']

Q: What payment methods are accepted?
   Retrieved: ['Payment Methods', 'Payment Methods']

Retrieval verified. Proceeding to graph construction.


## Part 2 — State Design

In [33]:
from typing import TypedDict, List, Optional

class CapstoneState(TypedDict):
    # Core fields
    question:     str
    messages:     List[dict]
    route:        str
    retrieved:    str
    sources:      List[str]
    tool_result:  str
    answer:       str
    faithfulness: float
    eval_retries: int
    # Domain-specific
    user_name:    Optional[str]

print('CapstoneState defined.')

CapstoneState defined.


## Part 3 — Node Functions

In [34]:
from langchain_groq import ChatGroq

def get_llm():
    return ChatGroq(
        model='llama-3.1-8b-instant',
        api_key=os.environ['GROQ_API_KEY'],
        temperature=0.1,
    )

print('LLM helper defined.')

LLM helper defined.


In [35]:

def memory_node(state: CapstoneState) -> dict:
    msgs = state.get('messages', [])
    msgs = msgs + [{'role': 'user', 'content': state['question']}]
    msgs = msgs[-6:]

    user_name = state.get('user_name')
    match = re.search(r'my name is ([A-Za-z]+)', state['question'], re.IGNORECASE)
    if match:
        user_name = match.group(1).capitalize()

    return {
        **state,
        'messages':  msgs,
        'user_name': user_name,
    }


_s = {'question': 'My name is Priya. What is the return policy?',
      'messages': [], 'route': '', 'retrieved': '', 'sources': [],
      'tool_result': '', 'answer': '', 'faithfulness': 1.0,
      'eval_retries': 0, 'user_name': None}
_out = memory_node(_s)
print('memory_node OK:', _out['user_name'], '|', len(_out['messages']), 'msgs')

memory_node OK: Priya | 1 msgs


In [36]:

def router_node(state: CapstoneState) -> dict:
    llm = get_llm()
    prompt = f"""You are a routing agent for an e-commerce customer support bot.

Classify the user question into EXACTLY ONE of these routes:
- retrieve  : question is about products, returns, shipping, refunds, warranty, payment, order tracking, or cancellation
- tool      : question asks for today's date or current time
- skip      : greeting, small talk, or anything not related to shopping or products

Reply with ONE word only — no punctuation, no explanation.

Question: {state['question']}
Route:"""
    raw = llm.invoke(prompt).content.strip().lower().split()[0]
    route = raw if raw in ('retrieve', 'tool', 'skip') else 'retrieve'
    return {**state, 'route': route}


_out = router_node({**_s, 'question': 'How do I return my headphones?'})
print('router_node OK:', _out['route'])

router_node OK: retrieve


In [37]:
def retrieval_node(state: CapstoneState) -> dict:
    docs = vectordb.similarity_search(state['question'], k=3)
    context_parts = []
    sources = []
    for d in docs:
        topic = d.metadata.get('topic', 'Info')
        context_parts.append(f'[{topic}]\n{d.page_content}')
        sources.append(topic)
    retrieved = '\n\n'.join(context_parts)
    return {**state, 'retrieved': retrieved, 'sources': sources}

_out = retrieval_node({**_s, 'question': 'What is the shipping cost?'})
print('retrieval_node OK:', _out['sources'])

retrieval_node OK: ['Shipping Policy', 'Shipping Policy', 'Shipping Policy']


In [38]:
def skip_retrieval_node(state: CapstoneState) -> dict:
    return {**state, 'retrieved': '', 'sources': []}

print('skip_retrieval_node OK')

skip_retrieval_node OK


In [39]:
def tool_node(state: CapstoneState) -> dict:
    try:
        now = datetime.datetime.now()
        result = f"Today is {now.strftime('%A, %d %B %Y')}. Current time is {now.strftime('%I:%M %p')} (IST)."
    except Exception as e:
        result = f'Could not retrieve date/time: {str(e)}'
    return {**state, 'tool_result': result, 'retrieved': '', 'sources': ['datetime']}

_out = tool_node(_s)
print('tool_node OK:', _out['tool_result'])

tool_node OK: Today is Tuesday, 21 April 2026. Current time is 05:31 PM (IST).


In [40]:
def answer_node(state: CapstoneState) -> dict:
    llm = get_llm()
    retrieved   = state.get('retrieved', '')
    tool_result = state.get('tool_result', '')
    user_name   = state.get('user_name', '')
    retries     = state.get('eval_retries', 0)

    name_part = f'The customer\'s name is {user_name}. Address them by name.' if user_name else ''

    retry_instruction = ''
    if retries > 0:
        retry_instruction = (
            f'\nThis is retry attempt {retries}. '
            'Your previous answer was not faithful to the context. '
            'Strictly use ONLY the information in the context below. '
            'If the answer is not in the context, say: "I don\'t have that information in my knowledge base."'
        )


    if retrieved:
        context_section = f'Context from knowledge base:\n{retrieved}'
    elif tool_result:
        context_section = f'Tool result:\n{tool_result}'
    else:

        context_section = ''

    if not context_section:
        prompt = f"""You are a friendly customer support assistant for ShopMax, an e-commerce store.
{name_part}
The customer sent a greeting or general message. Respond warmly and briefly.
Let them know you can help with products, shipping, returns, refunds, warranty, payment, and order tracking.
Do NOT make up any product or policy details.

Customer message: {state['question']}
"""
    else:
        prompt = f"""You are a customer support assistant for ShopMax, an e-commerce store.
{name_part}{retry_instruction}

RULES:
- Answer using ONLY the context below. Do NOT add outside knowledge.
- If the answer is not in the context, say exactly: "I don't have that information in my knowledge base."
- Be concise and direct. Do not repeat the question.
- Do not use emojis.

{context_section}

Customer question: {state['question']}
Answer:"""

    answer = llm.invoke(prompt).content.strip()
    return {**state, 'answer': answer}

print('answer_node defined.')

answer_node defined.


In [41]:
def eval_node(state: CapstoneState) -> dict:
    retrieved = state.get('retrieved', '')

    if not retrieved:
        return {**state, 'faithfulness': 1.0}

    llm = get_llm()
    prompt = f"""Rate how faithful the assistant answer is to the given context.
Score 1.0 = answer uses only context facts. Score 0.0 = answer invents information not in context.

Context:
{retrieved}

Answer:
{state['answer']}

Reply with a single decimal number between 0.0 and 1.0. Nothing else."""

    raw = llm.invoke(prompt).content.strip()
    try:
        score = float(re.findall(r'\d+\.?\d*', raw)[0])
        score = min(max(score, 0.0), 1.0)
    except Exception:
        score = 0.5

    retries = state.get('eval_retries', 0) + 1
    return {**state, 'faithfulness': score, 'eval_retries': retries}

print('eval_node defined.')

eval_node defined.


In [42]:
def save_node(state: CapstoneState) -> dict:
    msgs = state.get('messages', [])
    msgs = msgs + [{'role': 'assistant', 'content': state['answer']}]
    msgs = msgs[-6:]
    return {**state, 'messages': msgs}

print('save_node defined.')

save_node defined.


## Part 4 — Graph Assembly

In [43]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

def route_decision(state: CapstoneState) -> str:
    r = state.get('route', 'retrieve')
    if r == 'tool':     return 'tool'
    if r == 'skip':     return 'skip'
    return 'retrieve'

def eval_decision(state: CapstoneState) -> str:
    score   = state.get('faithfulness', 1.0)
    retries = state.get('eval_retries', 0)
    if score < 0.7 and retries < 2:
        return 'answer'   # retry
    return 'save'


def build_graph():
    memory = MemorySaver()
    g = StateGraph(CapstoneState)

    g.add_node('memory',    memory_node)
    g.add_node('router',    router_node)
    g.add_node('retrieval', retrieval_node)
    g.add_node('skip',      skip_retrieval_node)
    g.add_node('tool',      tool_node)
    g.add_node('answer',    answer_node)
    g.add_node('eval',      eval_node)
    g.add_node('save',      save_node)

    g.set_entry_point('memory')
    g.add_edge('memory', 'router')

    g.add_conditional_edges('router', route_decision, {
        'retrieve': 'retrieval',
        'skip':     'skip',
        'tool':     'tool',
    })

    g.add_edge('retrieval', 'answer')
    g.add_edge('skip',      'answer')
    g.add_edge('tool',      'answer')
    g.add_edge('answer',    'eval')

    g.add_conditional_edges('eval', eval_decision, {
        'answer': 'answer',
        'save':   'save',
    })

    g.add_edge('save', END)
    return g.compile(checkpointer=memory)

app = build_graph()
print('Graph compiled successfully.')

Graph compiled successfully.


## Part 5 — Testing

In [44]:
def ask(question: str, thread_id: str = 'test') -> dict:
    state = {
        'question':     question,
        'messages':     [],
        'route':        '',
        'retrieved':    '',
        'sources':      [],
        'tool_result':  '',
        'answer':       '',
        'faithfulness': 1.0,
        'eval_retries': 0,
        'user_name':    None,
    }
    config = {'configurable': {'thread_id': thread_id}}
    result = app.invoke(state, config)
    return result

print('ask() helper defined.')

ask() helper defined.


In [46]:
TEST_QUESTIONS = [
    ('What is the return policy?',                                    'domain'),
    ('How long does express shipping take?',                          'domain'),
    ('Tell me about the wireless headphones.',                        'domain'),
    ('What payment methods do you accept?',                           'domain'),
    ('How do I cancel my order?',                                     'domain'),
    ('Is the Smart Watch waterproof?',                                'domain'),
    ('How much does the USB-C charger cost?',                         'domain'),
    ('How do I claim a warranty?',                                    'domain'),
    #out-of-scope
    ('What is the weather in Mumbai today?',                          'red-team: out-of-scope'),
    #false premise
    ('I heard ShopMax offers free same-day delivery to all cities.',  'red-team: false premise'),
]

print('Test Run\n')
results = []
for question, category in TEST_QUESTIONS:
    r = ask(question, thread_id='test_run')
    score = r.get('faithfulness', 1.0)
    route = r.get('route', '?')
    answer = r.get('answer', '')
    status = 'PASS' if score >= 0.7 or not r.get('retrieved') else 'CHECK'
    results.append((question, category, route, score, status, answer))
    print(f'[{status}] [{category}] Route={route} | Faithfulness={score:.2f}')
    print(f'  Q: {question}')
    print(f'  A: {answer[:120]}...\n')

Test Run

[PASS] [domain] Route=retrieve | Faithfulness=1.00
  Q: What is the return policy?
  A: Customers may return any product within 7 days of delivery for a full refund, provided the item is unused, in its origin...

[PASS] [domain] Route=retrieve | Faithfulness=1.00
  Q: How long does express shipping take?
  A: Express delivery takes 1-2 business days....

[PASS] [domain] Route=retrieve | Faithfulness=0.90
  Q: Tell me about the wireless headphones.
  A: The ShopMax Wireless Headphones (Model WH-300) use Bluetooth 5.3 for a stable connection up to 10 metres. They have a ba...

[PASS] [domain] Route=retrieve | Faithfulness=1.00
  Q: What payment methods do you accept?
  A: We accept Credit cards (Visa, Mastercard, Amex, RuPay), debit cards (all major banks), UPI (Google Pay, PhonePe, Paytm, ...

[PASS] [domain] Route=retrieve | Faithfulness=1.00
  Q: How do I cancel my order?
  A: To cancel your order, go to My Orders in your account and click Cancel Order. You can do this for f

In [47]:
MEMORY_THREAD = 'memory_test_' + str(uuid.uuid4())

print('Memory Test\n')
q1 = ask('My name is Rahul.', thread_id=MEMORY_THREAD)
print('Q1:', q1['question'])
print('A1:', q1['answer'])
print()

q2 = ask('What is the return policy?', thread_id=MEMORY_THREAD)
print('Q2:', q2['question'])
print('A2:', q2['answer'])
print()

q3 = ask('Can you remind me what my name is?', thread_id=MEMORY_THREAD)
print('Q3:', q3['question'])
print('A3 (should mention Rahul):', q3['answer'])
print()
print('Memory test complete. Check that A3 references "Rahul".')

Memory Test

Q1: My name is Rahul.
A1: Hi Rahul, nice to meet you. Welcome to ShopMax's customer support. I'm here to help you with any questions or concerns you may have about our products, shipping, returns, refunds, warranty, payment, or tracking your order. How can I assist you today?

Q2: What is the return policy?
A2: Customers may return any product within 7 days of delivery for a full refund, provided the item is unused, in its original packaging, and all accessories and manuals are included.

Q3: Can you remind me what my name is?
A3 (should mention Rahul): Hello there. I'd be happy to help you with your query. Unfortunately, I'm a new chat assistant and I don't have access to your account information. However, you can easily access your account details by logging in to your ShopMax account. If you need assistance with anything else, I'm here to help with products, shipping, returns, refunds, warranty, payment, or order tracking. How can I assist you today?

Memory test comple

## Part 6 — RAGAS Baseline Evaluation

In [48]:
EVAL_PAIRS = [
    {
        'question':     'How many days do I have to return a product?',
        'ground_truth': 'Products can be returned within 7 days of delivery if unused and in original packaging.',
    },
    {
        'question':     'How much does express delivery cost?',
        'ground_truth': 'Express delivery costs Rs 99 regardless of order value and takes 1-2 business days.',
    },
    {
        'question':     'What is the battery life of the Smart Watch?',
        'ground_truth': 'The Smart Watch has 7 days of battery life in normal mode and 3 days with always-on display.',
    },
    {
        'question':     'What payment methods are accepted?',
        'ground_truth': 'ShopMax accepts credit cards, debit cards, UPI, net banking, Cash on Delivery, and ShopMax Wallet. EMI is available on orders above Rs 3,000.',
    },
    {
        'question':     'How do I cancel my order?',
        'ground_truth': 'Orders can be cancelled for free within 12 hours of placement if not yet dispatched by going to My Orders and clicking Cancel Order.',
    },
]


eval_data = {'question': [], 'answer': [], 'contexts': [], 'ground_truth': []}

print('Running RAGAS evaluation...\n')
for pair in EVAL_PAIRS:
    r = ask(pair['question'], thread_id='ragas_eval')
    eval_data['question'].append(pair['question'])
    eval_data['answer'].append(r.get('answer', ''))
    eval_data['contexts'].append([r.get('retrieved', '')])
    eval_data['ground_truth'].append(pair['ground_truth'])
    print(f"Q: {pair['question']}")
    print(f"A: {r.get('answer', '')[:100]}...\n")


try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    dataset = Dataset.from_dict(eval_data)
    ragas_result = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_precision])
    print('\n=== RAGAS Baseline Scores ===')
    print(ragas_result)
except Exception as e:

    print(f'RAGAS auto-eval failed ({e}). Using manual LLM faithfulness fallback.\n')
    llm = get_llm()
    scores = []
    for i, pair in enumerate(EVAL_PAIRS):
        ctx  = eval_data['contexts'][i][0]
        ans  = eval_data['answer'][i]
        gt   = pair['ground_truth']
        prompt = f"""Rate faithfulness of the answer to the context. Reply with a number 0.0-1.0 only.
Context: {ctx[:500]}
Answer: {ans}
Score:"""
        raw = llm.invoke(prompt).content.strip()
        try:
            s = float(re.findall(r'\d+\.?\d*', raw)[0])
        except:
            s = 0.5
        scores.append(s)
        print(f"  [{i+1}] Faithfulness: {s:.2f} | Q: {pair['question']}")
    print(f'\nMean manual faithfulness: {sum(scores)/len(scores):.2f}')

Running RAGAS evaluation...

Q: How many days do I have to return a product?
A: You have 7 days from the date of delivery to return a product....

Q: How much does express delivery cost?
A: Express delivery costs Rs 99 regardless of order value....

Q: What is the battery life of the Smart Watch?
A: The battery life of the Smart Watch is 7 days in normal mode, and 3 days with the always-on display ...

Q: What payment methods are accepted?
A: ShopMax accepts Credit cards (Visa, Mastercard, Amex, RuPay), debit cards (all major banks), UPI (Go...

Q: How do I cancel my order?
A: To cancel your order, go to My Orders in your account and click Cancel Order. You can do this for fr...

RAGAS auto-eval failed (The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable). Using manual LLM faithfulness fallback.



/tmp/ipykernel_11841/474822346.py:40: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/tmp/ipykernel_11841/474822346.py:40: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/tmp/ipykernel_11841/474822346.py:40: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness, answer_relevancy, context

  [1] Faithfulness: 0.90 | Q: How many days do I have to return a product?
  [2] Faithfulness: 0.90 | Q: How much does express delivery cost?
  [3] Faithfulness: 0.90 | Q: What is the battery life of the Smart Watch?
  [4] Faithfulness: 0.90 | Q: What payment methods are accepted?
  [5] Faithfulness: 0.90 | Q: How do I cancel my order?

Mean manual faithfulness: 0.90


## Part 7 — Streamlit UI (write app.py)

In [49]:
APP_CODE = '''
import os, re, uuid, datetime
from typing import TypedDict, List, Optional
import streamlit as st
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# ---- State ----
class CapstoneState(TypedDict):
    question:     str
    messages:     List[dict]
    route:        str
    retrieved:    str
    sources:      List[str]
    tool_result:  str
    answer:       str
    faithfulness: float
    eval_retries: int
    user_name:    Optional[str]

# ---- Knowledge Base ----
DOCUMENTS = [
    {"id": "doc_001", "topic": "Wireless Headphones",
     "text": "The ShopMax Wireless Headphones (Model WH-300) use Bluetooth 5.3 for a stable, low-latency connection up to 10 metres. Battery life is 20 hours on a single charge, with a 2-hour fast-charge via USB-C. The headphones include active noise cancellation (ANC), a built-in microphone for calls, and 40mm drivers for rich bass. Weight is 250g. They are foldable and come with a hard carrying case. Available colours: Black, White, Navy Blue. Price: Rs 1,999. Covered by a 1-year manufacturer warranty against hardware defects. Compatible with iOS, Android, Windows, and macOS. Not waterproof."},
    {"id": "doc_002", "topic": "Smart Watch",
     "text": "The ShopMax Smart Watch (Model SW-200) tracks steps, calories, sleep, heart rate, and blood oxygen (SpO2). It has a 1.4-inch AMOLED display with always-on mode. Battery life is 7 days in normal mode, 3 days with always-on display enabled. It is IP68 waterproof -- safe for swimming up to 50 metres. Compatible with iOS 12+ and Android 8+. Includes 20+ workout modes (running, cycling, yoga, swimming). Built-in GPS for outdoor runs. Supports notifications for calls, SMS, and apps. Strap is interchangeable (22mm standard). Price: Rs 2,999. 1-year warranty."},
    {"id": "doc_003", "topic": "Mechanical Keyboard",
     "text": "The ShopMax Mechanical Keyboard (Model KB-75) is a tenkeyless (TKL) wired keyboard with Cherry MX Red switches -- linear, quiet, and ideal for both typing and gaming. It has per-key RGB backlighting with 15 preset lighting modes. The aluminium top plate and PBT double-shot keycaps make it durable and resistant to shine. N-key rollover (NKRO) ensures every keypress is registered simultaneously. Connection: USB-A braided cable (1.8m). Layout: US ANSI. Compatible with Windows 10/11 and macOS. No driver software required. Price: Rs 3,499. 2-year warranty. Available in Space Grey only."},
    {"id": "doc_004", "topic": "Portable Bluetooth Speaker",
     "text": "The ShopMax Blast Speaker (Model BS-50) delivers 20W stereo sound with dual 10W drivers and a passive radiator for deep bass. Battery lasts 12 hours at 70% volume. It is IPX7 waterproof -- fully submersible up to 1 metre for 30 minutes. Bluetooth 5.0 range is up to 15 metres. Also supports wired AUX input (3.5mm). Has a built-in microphone for speakerphone calls. Charges in 3 hours via USB-C. Dimensions: 18cm x 7cm. Weight: 600g. Colours: Red, Black, Forest Green. Price: Rs 2,499. 1-year warranty. Not for underwater use beyond 1 metre."},
    {"id": "doc_005", "topic": "USB-C Fast Charger",
     "text": "The ShopMax 65W USB-C GaN Charger (Model FC-65) supports USB Power Delivery 3.0 and charges laptops, tablets, and phones. It has two ports: one USB-C (65W max) and one USB-A (18W max, Quick Charge 3.0). The GaN technology makes it 40% smaller than traditional chargers of the same wattage. Charges an iPhone 15 to 50% in 30 minutes. Charges a MacBook Air in under 2 hours. Universal voltage (100-240V) -- suitable for international travel. Cable not included. Price: Rs 1,299. 1-year warranty. Colour: White."},
    {"id": "doc_006", "topic": "Laptop Stand",
     "text": "The ShopMax ErgoRise Laptop Stand (Model LS-20) is made from anodised aluminium and supports laptops from 10 to 16 inches weighing up to 10kg. Height is adjustable across 6 levels (10cm to 20cm) and the angle can be set from 15 to 45 degrees. Folds flat in seconds for portability -- packed size is 28cm x 22cm x 1.5cm, weight 500g. Silicone pads grip the desk and protect the laptop from scratches. Does not include a USB hub. Compatible with all brands. Price: Rs 899. 1-year warranty. Available in Silver and Space Grey."},
    {"id": "doc_007", "topic": "Return Policy",
     "text": "Customers may return any product within 7 days of delivery for a full refund, provided the item is unused, in its original packaging, and all accessories and manuals are included. To initiate a return, log in to your ShopMax account, go to Orders, select the item, and click Return. A pickup will be arranged within 1-2 business days at no cost to the customer. Items that are damaged due to customer misuse, have missing parts, or are returned after 7 days are not eligible for a refund. Opened software, downloaded digital products, and personalised/custom items cannot be returned. Refunds are processed within 5-7 business days of the return being received and verified."},
    {"id": "doc_008", "topic": "Shipping Policy",
     "text": "ShopMax ships to all major cities and towns across India. Standard delivery takes 3-5 business days and is free for orders above Rs 499. For orders below Rs 499, a flat shipping fee of Rs 49 applies. Express delivery (1-2 business days) costs Rs 99 regardless of order value. Same-day delivery is available in Bangalore, Mumbai, Delhi, Hyderabad, and Chennai for orders placed before 11am. Same-day delivery costs Rs 149. Orders are dispatched Monday to Saturday (excluding public holidays). A tracking link is sent via SMS and email once the order ships. International shipping is not currently available."},
    {"id": "doc_009", "topic": "Refund Policy",
     "text": "Once a returned item is received and verified by our warehouse team, the refund is processed within 5-7 business days. For orders paid by credit or debit card, the refund appears on the original card. For UPI and net banking payments, the refund is credited to the original payment account. Cash on Delivery orders are refunded via bank transfer -- customers must provide their bank account number and IFSC code. ShopMax Wallet refunds are processed within 24 hours. Customers receive an email confirmation when the refund is initiated. If the refund is not received within 7 business days, contact support with your order ID."},
    {"id": "doc_010", "topic": "Warranty Policy",
     "text": "All ShopMax products include a manufacturer warranty covering hardware defects. Most products carry a 1-year warranty; the Mechanical Keyboard (KB-75) carries a 2-year warranty. Warranty claims must be submitted within the warranty period by contacting support at support@shopmax.in or calling 1800-123-4567 (toll-free). The warranty covers: dead-on-arrival (DOA) units, manufacturing defects, and component failure under normal use. The warranty does NOT cover: physical damage, water damage (unless the product is rated waterproof), damage from incorrect voltage, and damage from unauthorised repair. Proof of purchase (order ID or invoice) is required. Approved claims result in a free repair or replacement."},
    {"id": "doc_011", "topic": "Payment Methods",
     "text": "ShopMax accepts: credit cards (Visa, Mastercard, Amex, RuPay), debit cards (all major banks), UPI (Google Pay, PhonePe, Paytm, BHIM), net banking (50+ banks), Cash on Delivery (COD) for orders up to Rs 10,000, and ShopMax Wallet. EMI is available on orders above Rs 3,000 via eligible credit cards (3, 6, 9, and 12-month tenures). Buy Now Pay Later (BNPL) is available through ZestMoney and LazyPay. All transactions are secured with 256-bit SSL encryption. Prices are inclusive of GST (18% for electronics)."},
    {"id": "doc_012", "topic": "Order Tracking and Cancellation",
     "text": "Once an order is placed, a confirmation email and SMS are sent immediately. After dispatch, a tracking link from our courier partner (Delhivery, Ekart, or BlueDart) is shared via SMS and email. To track your order, visit shopmax.in/track or use the ShopMax app and enter your order ID. Orders can be cancelled for free within 12 hours of placement if not yet dispatched. To cancel, go to My Orders in your account and click Cancel Order. Orders that are already dispatched cannot be cancelled -- wait for delivery and then raise a return request. Refunds for cancelled pre-dispatch orders are processed within 24 hours. Contact support@shopmax.in or call 1800-123-4567 for help."},
]

# ---- Load VectorDB (cached) ----
@st.cache_resource(show_spinner="Loading knowledge base...")
def load_vectordb():
    emb = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    texts = [d["text"]  for d in DOCUMENTS]
    metas = [{"topic": d["topic"], "id": d["id"]} for d in DOCUMENTS]
    return Chroma.from_texts(texts=texts, embedding=emb, metadatas=metas)

# ---- LLM ----
def get_llm():
    return ChatGroq(model="llama-3.1-8b-instant", api_key=os.environ["GROQ_API_KEY"], temperature=0.1)

# ---- Nodes ----
def memory_node(state):
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    msgs = msgs[-6:]
    user_name = state.get("user_name")
    m = re.search(r"my name is ([A-Za-z]+)", state["question"], re.IGNORECASE)
    if m:
        user_name = m.group(1).capitalize()
    return {**state, "messages": msgs, "user_name": user_name}

def router_node(state):
    llm = get_llm()
    prompt = f"""You are a routing agent for an e-commerce customer support bot.
Classify the question into ONE of: retrieve | tool | skip
- retrieve: products, returns, shipping, refunds, warranty, payment, order tracking, cancellation
- tool: today\'s date or current time
- skip: greetings, small talk, unrelated topics
Reply with ONE word only.
Question: {state[\'question\']}
Route:"""
    raw = llm.invoke(prompt).content.strip().lower().split()[0]
    route = raw if raw in ("retrieve", "tool", "skip") else "retrieve"
    return {**state, "route": route}

def retrieval_node(state):
    db = load_vectordb()
    docs = db.similarity_search(state["question"], k=3)
    parts = []
    sources = []
    for d in docs:
        topic = d.metadata.get("topic", "Info")
        parts.append(f"[{topic}]\\n{d.page_content}")
        sources.append(topic)
    return {**state, "retrieved": "\\n\\n".join(parts), "sources": sources}

def skip_retrieval_node(state):
    return {**state, "retrieved": "", "sources": []}

def tool_node(state):
    try:
        now = datetime.datetime.now()
        result = f"Today is {now.strftime(\'%A, %d %B %Y\')}. Current time is {now.strftime(\'%I:%M %p\')} (IST)."
    except Exception as e:
        result = f"Could not retrieve date/time: {e}"
    return {**state, "tool_result": result, "retrieved": "", "sources": ["datetime"]}

def answer_node(state):
    llm = get_llm()
    retrieved   = state.get("retrieved", "")
    tool_result = state.get("tool_result", "")
    user_name   = state.get("user_name", "")
    retries     = state.get("eval_retries", 0)
    name_part   = f"The customer\'s name is {user_name}. Address them by name." if user_name else ""
    retry_instr = (f"\\nRetry {retries}: previous answer was not faithful. Use ONLY context. If not in context, say so."
                   if retries > 0 else "")
    if retrieved:
        ctx = f"Context from knowledge base:\\n{retrieved}"
    elif tool_result:
        ctx = f"Tool result:\\n{tool_result}"
    else:
        ctx = ""
    if not ctx:
        prompt = f"""You are a friendly customer support assistant for ShopMax. {name_part}
Respond warmly and briefly. Let them know you can help with products, shipping, returns, refunds, warranty, payment, and order tracking.
Customer message: {state[\'question\']}"""
    else:
        prompt = f"""You are a customer support assistant for ShopMax. {name_part}{retry_instr}
RULES: Answer using ONLY the context. If not in context say: "I don\'t have that information in my knowledge base."
Be concise. Do not use emojis.
{ctx}
Customer question: {state[\'question\']}
Answer:"""
    answer = llm.invoke(prompt).content.strip()
    return {**state, "answer": answer}

def eval_node(state):
    if not state.get("retrieved", ""):
        return {**state, "faithfulness": 1.0}
    llm = get_llm()
    prompt = f"""Rate faithfulness of the answer to the context. Reply with a decimal 0.0-1.0 only.
Context: {state[\'retrieved\'][:600]}
Answer: {state[\'answer\']}
Score:"""
    raw = llm.invoke(prompt).content.strip()
    try:
        score = float(re.findall(r"\\d+\\.?\\d*", raw)[0])
        score = min(max(score, 0.0), 1.0)
    except Exception:
        score = 0.5
    return {**state, "faithfulness": score, "eval_retries": state.get("eval_retries", 0) + 1}

def save_node(state):
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "assistant", "content": state["answer"]}]
    return {**state, "messages": msgs[-6:]}

def route_decision(state):
    r = state.get("route", "retrieve")
    return r if r in ("tool", "skip") else "retrieve"

def eval_decision(state):
    if state.get("faithfulness", 1.0) < 0.7 and state.get("eval_retries", 0) < 2:
        return "answer"
    return "save"

# ---- Build Graph (cached) ----
@st.cache_resource(show_spinner="Building assistant...")
def build_graph():
    mem = MemorySaver()
    g   = StateGraph(CapstoneState)
    g.add_node("memory",    memory_node)
    g.add_node("router",    router_node)
    g.add_node("retrieval", retrieval_node)
    g.add_node("skip",      skip_retrieval_node)
    g.add_node("tool",      tool_node)
    g.add_node("answer",    answer_node)
    g.add_node("eval",      eval_node)
    g.add_node("save",      save_node)
    g.set_entry_point("memory")
    g.add_edge("memory", "router")
    g.add_conditional_edges("router", route_decision, {"retrieve": "retrieval", "skip": "skip", "tool": "tool"})
    g.add_edge("retrieval", "answer")
    g.add_edge("skip",      "answer")
    g.add_edge("tool",      "answer")
    g.add_edge("answer",    "eval")
    g.add_conditional_edges("eval", eval_decision, {"answer": "answer", "save": "save"})
    g.add_edge("save", END)
    return g.compile(checkpointer=mem)

# ---- Streamlit UI ----
st.set_page_config(page_title="ShopMax Support", layout="centered")

graph    = build_graph()
vectordb = load_vectordb()

if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []
if "bot_state" not in st.session_state:
    st.session_state.bot_state = {
        "question": "", "messages": [], "route": "", "retrieved": "",
        "sources": [], "tool_result": "", "answer": "",
        "faithfulness": 1.0, "eval_retries": 0, "user_name": None,
    }

with st.sidebar:
    st.title("ShopMax Support")
    st.caption("E-commerce FAQ Assistant")
    st.divider()
    if st.button("New Conversation"):
        st.session_state.chat_history = []
        st.session_state.thread_id    = str(uuid.uuid4())
        st.session_state.bot_state    = {
            "question": "", "messages": [], "route": "", "retrieved": "",
            "sources": [], "tool_result": "", "answer": "",
            "faithfulness": 1.0, "eval_retries": 0, "user_name": None,
        }
        st.rerun()
    st.divider()
    st.caption("Can help with: products, shipping, returns, refunds, warranty, payment, order tracking.")
    st.caption("For specific order issues, contact support@shopmax.in or call 1800-123-4567.")

st.title("ShopMax Customer Support")
st.caption("Ask about products, shipping, returns, warranty, or payment.")
st.divider()

for msg in st.session_state.chat_history:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

user_input = st.chat_input("Type your question here...")
if user_input:
    st.session_state.chat_history.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.write(user_input)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            state = st.session_state.bot_state.copy()
            state["question"]     = user_input
            state["eval_retries"] = 0
            try:
                config = {"configurable": {"thread_id": st.session_state.thread_id}}
                result = graph.invoke(state, config)
                answer = result.get("answer", "Something went wrong. Please try again.")
                st.session_state.bot_state = result
            except Exception as e:
                answer = f"Error: {e}"
        st.write(answer)

    st.session_state.chat_history.append({"role": "assistant", "content": answer})
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(APP_CODE)
print('app.py written successfully.')

app.py written successfully.


## Part 7 — Launch Streamlit via ngrok

In [50]:
from pyngrok import ngrok, conf

if not NGROK_TOKEN:
    print('ERROR: NGROK_AUTH_TOKEN not found in Colab Secrets. Cannot launch UI.')
else:
    conf.get_default().auth_token = NGROK_TOKEN

    # Kill any existing processes
    ngrok.kill()
    subprocess.run(['pkill', '-f', 'streamlit'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)

    # Start Streamlit
    subprocess.Popen(
        ['streamlit', 'run', 'app.py',
         '--server.port', '8501',
         '--server.enableCORS', 'false',
         '--server.enableXsrfProtection', 'false'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(5)

    # Open ngrok tunnel
    public_url = ngrok.connect(8501)
    print('\nShopMax Support Bot is live at:\n')
    print(public_url.public_url)
    print('\nOpen the URL above in your browser.')


ShopMax Support Bot is live at:

https://tribune-zone-unrevised.ngrok-free.dev

Open the URL above in your browser.


In [51]:
# Run this cell to stop the Streamlit server and ngrok tunnel
!pkill -f streamlit
!pkill ngrok
print('Server stopped.')

Server stopped.
